# 01 — H&M data and temporal structure

This walkthrough uses the real, fixed 50,000-customer cohort. Code stays in `src/ranking`; the notebook is for exploration. `make prepare` and `make run` have already produced the local tables and aggregate report.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ranking.data import load_tables, temporal_snapshot
tx, customers, articles = load_tables(ROOT / 'data/processed/hm_sample')
print('Transactions:', len(tx), 'Customers:', len(customers), 'Articles:', len(articles))

## Observed customer behavior

Repeated rows are retained. Purchase frequency and item diversity describe recorded transactions; absence of a purchase does not establish a dislike.

In [ ]:
behavior = tx.groupby('customer_id').agg(purchases=('article_id', 'size'), unique_articles=('article_id', 'nunique'))
behavior.describe(percentiles=[.25, .5, .75, .9, .99])

In [ ]:
weekly = pd.read_csv(ROOT / 'reports/hm_sample/eda/weekly_activity.csv', parse_dates=['week']).set_index('week')
weekly.plot(subplots=True, figsize=(11, 7), title='Weekly activity — fixed customer sample')

## Half-open temporal windows

The cutoff itself belongs to the target week. A later transaction must never enter the historical features or retrieval model.

In [ ]:
history, target = temporal_snapshot(tx, '2020-09-16', horizon_days=7, history_days=120)
print('History:', history.t_dat.min(), 'through', history.t_dat.max())
print('Target:', target.t_dat.min(), 'through', target.t_dat.max())
print('Target purchasers:', target.customer_id.nunique())

In [ ]:
import json
summary = json.loads((ROOT / 'reports/hm_sample/snapshots/2020-09-16/summary.json').read_text())
summary

Next: inspect candidate recall and the strong recent-purchase baseline in notebook 02. The formal test results are already frozen; use a new development period for new tuning.